## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
Eto ilay references rehetra no apetraka

---

# Introduction : Régression LASSO

## Qu'est-ce que le LASSO ?

Le **LASSO** (*Least Absolute Shrinkage and Selection Operator*) est une méthode de régression linéaire régularisée. Il s'agit d'une extension de la régression linéaire classique qui ajoute une **pénalité L1** sur les coefficients.

## À quoi sert-il ?

Le LASSO sert à :
- **Réduire l'overfitting** en pénalisant les grands coefficients
- **Sélectionner automatiquement les variables** : contrairement à Ridge, le LASSO met exactement certains coefficients à **zéro** (sparse solution)
- **Améliorer l'interprétabilité** du modèle en ne gardant que les features les plus importantes

## Formulation mathématique

On cherche à minimiser :

$$\min_{w, b} \underbrace{\frac{1}{N} \sum_{i=1}^{N} (y_i - X_i^\top w - b)^2}_{\text{MSE (fidélité aux données)}} + \underbrace{\alpha \|w\|_1}_{\text{pénalité L1}}$$

où :
- $X \in \mathbb{R}^{N \times d}$ : matrice des features
- $y \in \mathbb{R}^N$ : vecteur des cibles
- $w \in \mathbb{R}^d$ : vecteur des poids (coefficients)
- $b \in \mathbb{R}$ : biais (intercept)
- $\alpha > 0$ : coefficient de régularisation (plus $\alpha$ est grand, plus la solution est sparse)
- $\|w\|_1 = \sum_{j=1}^d |w_j|$ : norme L1

## Pourquoi la norme L1 produit-elle de la sparsité ?

La norme L1 a des **coins** en zéro (contrairement à L2 qui est lisse). Le gradient de la pénalité L1 pousse exactement les petits coefficients à **zéro**, ce qui permet la sélection de variables.

## Difficulté : non-différentiabilité

La norme $\|w\|_1$ **n'est pas différentiable** en $w_j = 0$. Cela impose d'utiliser des méthodes d'optimisation spéciales :
1. **Subgradient Descent** : utilise un sous-gradient de $|w_j|$
2. **Proximal Gradient Descent (ISTA)** : sépare la partie lisse et la partie non-lisse
3. **Projected Gradient Descent** : décompose $w = w^+ - w^-$ avec $w^+, w^- \geq 0$
4. **Coordinate Descent** : optimise une coordonnée à la fois avec solution analytique

---

In [ ]:
from random import randrange
import numpy as np
from sklearn.metrics import mean_squared_error, log_loss
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import normalize


def grad_check_sparse(f, x, analytic_grad, num_checks=10, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
data = load_diabetes()
X, y = data.data, data.target

---
## 1. Fonction de perte MSE vectorisée

### Qu'est-ce que c'est ?

`mse_loss_vectorized` calcule la **Mean Squared Error (MSE)** et ses gradients analytiques par rapport aux poids $w$ et au biais $b$, **sans boucle** (opérations matricielles pures) et **sans régularisation**.

### Formule mathématique

**Loss :**
$$\mathcal{L}(w, b) = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i)^2 = \frac{1}{N} \|Xw + b\mathbf{1} - y\|^2$$

**Prédiction :**
$$\hat{y} = Xw + b \quad \in \mathbb{R}^N$$

**Résidus :**
$$r = \hat{y} - y \quad \in \mathbb{R}^N$$

**Gradient par rapport à $w$ :**
$$\frac{\partial \mathcal{L}}{\partial w} = \frac{2}{N} X^\top r$$

**Gradient par rapport à $b$ :**
$$\frac{\partial \mathcal{L}}{\partial b} = \frac{2}{N} \sum_{i=1}^N r_i = \frac{2}{N} \mathbf{1}^\top r$$

### Dérivation

En posant $r = Xw + b - y$, on a $\mathcal{L} = \frac{1}{N} r^\top r$, donc :
$$\frac{\partial \mathcal{L}}{\partial w} = \frac{2}{N} X^\top r, \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{2}{N} \mathbf{1}^\top r$$

In [ ]:
def mse_loss_vectorized(w, b, X, y):
    """
    MSE loss function WITHOUT FOR LOOPs , NO REGULARIZATION
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    - gradient with respect to bias b
    """
    loss = 0.0
    dw = np.zeros_like(w)
    
    N = X.shape[0]
    y_pred = X @ w + b
    residuals = y_pred - y
    loss = np.sum(residuals ** 2) / N
    dw = 2 * X.T @ residuals / N
    db = 2 * np.sum(residuals) / N
    
    return loss, dw, np.array(db).reshape(1,)

---
## 2. Opérateur de seuillage doux (Soft Threshold)

### Qu'est-ce que c'est ?

La fonction `soft_threshold` est l'**opérateur proximal** associé à la norme L1. C'est la brique fondamentale de plusieurs algorithmes LASSO.

### Formule mathématique

$$\text{soft\_threshold}(x, \delta) = \text{sign}(x) \cdot \max(|x| - \delta,\ 0)$$

Ce qui revient à :
$$\text{soft\_threshold}(x, \delta) = \begin{cases} x - \delta & \text{si } x > \delta \\ 0 & \text{si } |x| \leq \delta \\ x + \delta & \text{si } x < -\delta \end{cases}$$

### Interprétation

- Si $|x| \leq \delta$ → la valeur est **mise à zéro** (d'où la sparsité du LASSO)
- Si $|x| > \delta$ → la valeur est **réduite** de $\delta$ vers zéro

### Origine mathématique

C'est la solution exacte du problème proximal :
$$\text{prox}_{\delta \|\cdot\|_1}(x) = \arg\min_u \left\{ \frac{1}{2}\|u - x\|^2 + \delta \|u\|_1 \right\}$$

In [ ]:
def soft_threshold(x, delta):
    return np.sign(x) * np.maximum(np.abs(x) - delta, 0)

---
# Lasso Subgradient Descent

## Qu'est-ce que c'est ?

La **descente de sous-gradient** est l'extension naturelle de la descente de gradient pour les fonctions **non-différentiables** comme $\|w\|_1$. On remplace le gradient par un **sous-gradient** de la norme L1.

## Mathématiques : Sous-gradient de la norme L1

Le **sous-gradient** de $f(w) = \|w\|_1 = \sum_j |w_j|$ est :

$$\partial |w_j| = \begin{cases} \{+1\} & \text{si } w_j > 0 \\ [-1, +1] & \text{si } w_j = 0 \\ \{-1\} & \text{si } w_j < 0 \end{cases}$$

En pratique, on choisit le sous-gradient $g_j = \text{sign}(w_j)$ (avec $\text{sign}(0) = 0$).

## Fonction objectif LASSO avec sous-gradient

$$\mathcal{L}_{\text{lasso}}(w, b) = \underbrace{\frac{1}{N}\|Xw + b - y\|^2}_{\text{MSE}} + \alpha \|w\|_1$$

Le sous-gradient total par rapport à $w$ est :
$$g_w = \underbrace{\frac{2}{N} X^\top (Xw + b - y)}_{\nabla_w \text{MSE}} + \alpha \cdot \text{sign}(w)$$

## Algorithme : Stochastic Subgradient Descent

```
Initialiser w ← petit bruit aléatoire, b ← 0
Pour t = 1, 2, ..., T :
    1. Tirer un mini-batch aléatoire {(x_i, y_i)} de taille batch_size
    2. Calculer le sous-gradient : g_w = (2/B) X_batch^T (X_batch w + b - y_batch) + α·sign(w)
    3. Calculer le sous-gradient pour b : g_b = (2/B) Σ(x_i w + b - y_i)
    4. Mettre à jour : w ← w - η · g_w
                       b ← b - η · g_b
```

## Limitations

- La descente de sous-gradient **converge plus lentement** que le gradient proximal
- Elle ne garantit **pas** que les coefficients soient exactement zéro (solution approximative)
- Nécessite un grand nombre d'itérations pour converger

In [ ]:
def l1_subgradient(w):
    """
    Subgradient of the L1 loss
    """
    dw = np.zeros_like(w)
    dw = np.sign(w)
    return dw
    

def lasso_subgradient_mse_loss_vectorized(w, b, X, y, alpha):
    """
    MSE loss function adding the subgradient for w
    """
    loss, dw, db = mse_loss_vectorized(w, b, X, y)
    # Add the subgradient to dw
    dw = dw + alpha * l1_subgradient(w)
    return loss, dw, db

In [ ]:
class LassolSubgradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # YOUR CODE HERE
            self.w -= learning_rate * dw
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
    
    
    def predict(self, X):
        return X @ self.w + self.b

    def loss(self, X_batch, y_batch):
        return lasso_subgradient_mse_loss_vectorized(self.w, self.b, X_batch, y_batch, self.alpha)

In [ ]:
model = LassolSubgradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

In [ ]:
model = LassolSubgradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

---
# Lasso Proximal Gradient Descent (ISTA)

## Qu'est-ce que c'est ?

La **descente de gradient proximal** (aussi appelée **ISTA** : Iterative Soft Thresholding Algorithm) est une méthode plus efficace que le sous-gradient. Elle exploite la structure **composite** de la fonction objectif LASSO : une partie lisse (MSE) + une partie non-lisse (L1).

## Idée fondamentale

On sépare le problème en deux :
- **Partie lisse** $f(w) = \frac{1}{N}\|Xw + b - y\|^2$ → on fait un pas de gradient
- **Partie non-lisse** $g(w) = \alpha\|w\|_1$ → on applique l'opérateur proximal (soft threshold)

## Mathématiques

L'**opérateur proximal** de $g(w) = \alpha\|w\|_1$ avec pas $\eta$ est :
$$\text{prox}_{\eta g}(v) = \arg\min_u \left\{\frac{1}{2}\|u - v\|^2 + \eta\alpha\|u\|_1\right\} = \text{soft\_threshold}(v,\ \eta\alpha)$$

## Algorithme ISTA (Proximal SGD)

```
Initialiser w ← petit bruit aléatoire, b ← 0
Pour t = 1, 2, ..., T :
    1. Tirer un mini-batch aléatoire de taille batch_size
    2. Calculer le gradient MSE : dw = (2/B) X_batch^T (X_batch w + b - y_batch)
    3. Pas de gradient sur la partie lisse : w_half = w - η · dw
    4. Appliquer l'opérateur proximal (soft threshold) :
          w ← soft_threshold(w_half, η·α)
    5. Mettre à jour le biais : b ← b - η · db
```

## Pourquoi c'est mieux que le sous-gradient ?

- **Convergence plus rapide** : $O(1/t)$ vs $O(1/\sqrt{t})$ pour le sous-gradient
- **Sparsité exacte** : le soft threshold met des coefficients **exactement à zéro**
- **Séparation propre** entre la partie gradient et la partie proximale

In [ ]:
class LassoProximalGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # PROJECT THE GRADIENT FOR w USING soft_threshold
            # YOUR CODE HERE
            self.w = soft_threshold(self.w - learning_rate * dw, learning_rate * self.alpha)
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        return X @ self.w + self.b

    def loss(self, X_batch, y_batch):
        return mse_loss_vectorized(self.w, self.b, X_batch, y_batch)

In [ ]:
model = LassoProximalGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

In [ ]:
model = LassoProximalGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

---
# Lasso Projected Gradient Descent

## Qu'est-ce que c'est ?

La **descente de gradient projeté** transforme le problème LASSO (non-convexe à cause de la norme L1) en un problème **d'optimisation sous contraintes de non-négativité**, qui est convexe et différentiable partout.

## Idée : Décomposition de $w$

On décompose chaque coefficient $w_j$ en sa partie **positive** et **négative** :
$$w_j = w_j^+ - w_j^-, \quad w_j^+ \geq 0, \quad w_j^- \geq 0$$

Ainsi : $|w_j| = w_j^+ + w_j^-$ et la contrainte $w_j^+, w_j^- \geq 0$ s'impose naturellement.

## Nouveau problème d'optimisation

On réécrit le LASSO comme :
$$\min_{w^+, w^- \geq 0} \frac{1}{N}\|X(w^+ - w^-) + b - y\|^2 + \alpha \mathbf{1}^\top(w^+ + w^-)$$

Le gradient par rapport à $w^+$ et $w^-$ est :
$$\frac{\partial \mathcal{L}}{\partial w^+} = \frac{2}{N} X^\top r + \alpha$$
$$\frac{\partial \mathcal{L}}{\partial w^-} = -\frac{2}{N} X^\top r + \alpha$$

où $r = X(w^+ - w^-) + b - y$.

## Algorithme

```
Initialiser w_p, w_n ← petit bruit positif, b ← 0
Pour t = 1, 2, ..., T :
    1. Tirer un mini-batch
    2. Calculer dw = gradient MSE par rapport à w = w_p - w_n
    3. Mettre à jour et projeter sur le cône positif :
          w_p ← max(w_p - η·(dw + α), 0)   [projection sur R+]
          w_n ← max(w_n + η·(dw - α), 0)   [projection sur R+]
          b   ← b - η·db
    4. Prédiction : ŷ = X(w_p - w_n) + b
```

## Projection sur le cône positif

La projection $\Pi_{\mathbb{R}_+}(v) = \max(v, 0)$ assure que $w^+, w^- \geq 0$ à chaque itération.

## Avantage

- Le problème devient **différentiable partout** (pas besoin de sous-gradient)
- La contrainte de non-négativité est gérée par la **projection** simple $\max(\cdot, 0)$

In [ ]:
class LassoProjectedGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w_p = None
        self.w_n = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w_p is None: # Initialization
            self.w_p = 0.001 * np.random.randn(d)
            self.w_n = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # Project for w_p and w_n
            # YOUR CODE HERE
            self.w_p = np.maximum(self.w_p - learning_rate * (dw + self.alpha), 0)
            self.w_n = np.maximum(self.w_n + learning_rate * (dw - self.alpha), 0)
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        w = self.w_p - self.w_n
        return X @ w + self.b

    def loss(self, X_batch, y_batch):
        w = self.w_p - self.w_n
        return mse_loss_vectorized(w, self.b, X_batch, y_batch)

In [ ]:
model = LassoProjectedGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

In [ ]:
model = LassoProjectedGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

---
# Lasso Coordinate Descent (no intercept)

## Qu'est-ce que c'est ?

La **descente par coordonnée** (Coordinate Descent) est l'algorithme le plus utilisé en pratique pour le LASSO (c'est ce que fait scikit-learn par défaut). Elle optimise **une coordonnée $w_j$ à la fois** en gardant toutes les autres fixées, ce qui admet une **solution analytique exacte** grâce au soft threshold.

## Idée fondamentale

Pour chaque coordonnée $j$, on minimise $\mathcal{L}(w_j)$ en fixant tous les autres $w_k$ ($k \neq j$). On obtient un sous-problème 1D avec solution fermée.

## Dérivation mathématique

On définit le **résidu partiel** (résidu quand on retire la contribution de la coordonnée $j$) :
$$r^{(j)} = y - \sum_{k \neq j} X_{:,k} \cdot w_k = y - Xw + X_{:,j} w_j$$

La MSE restreinte à $w_j$ devient :
$$\mathcal{L}(w_j) = \frac{1}{N}\|X_{:,j} w_j - r^{(j)}\|^2 + \alpha |w_j| + \text{const}$$

En développant :
$$\mathcal{L}(w_j) = \frac{\|X_{:,j}\|^2}{N} w_j^2 - \frac{2\rho_j}{N} w_j + \alpha|w_j| + \text{const}$$

où la **corrélation partielle** est :
$$\rho_j = X_{:,j}^\top r^{(j)}$$

## Solution analytique : mise à jour de la coordonnée

La solution exacte est donnée par le **soft threshold** :

$$w_j^* = \text{soft\_threshold}\left(\frac{\rho_j}{\|X_{:,j}\|^2},\ \frac{\alpha}{\|X_{:,j}\|^2}\right)$$

C'est-à-dire :
$$w_j^* = \begin{cases} \frac{\rho_j - \alpha}{\|X_{:,j}\|^2} & \text{si } \rho_j > \alpha \\ 0 & \text{si } |\rho_j| \leq \alpha \\ \frac{\rho_j + \alpha}{\|X_{:,j}\|^2} & \text{si } \rho_j < -\alpha \end{cases}$$

## Algorithme complet

```
Initialiser w ← 0
Précalculer c_j = ||X_{:,j}||^2  pour tout j

Pour t = 1, 2, ..., num_iters :
    Pour j = 0, 1, ..., d-1 :
        1. Calculer le résidu partiel : r_j = y - Xw + X_{:,j} * w_j
        2. Calculer la corrélation   : rho_j = X_{:,j}^T . r_j
        3. Mettre à jour             : w_j = soft_threshold(rho_j / c_j, alpha / c_j)
```

## Pourquoi c'est l'algorithme le plus efficace ?

- **Solution analytique exacte** à chaque étape → pas d'hyperparamètre learning rate
- **Convergence garantie** vers l'optimum global (problème convexe)
- **Sparsité exacte** : les $w_j$ deviennent exactement zéro quand $|\rho_j| \leq \alpha$
- Complexité par itération : $O(N \cdot d)$

In [ ]:
class LassoCoordinateDescent():
    def __init__(self, alpha=0.1):
        self.w = None
        self.alpha = alpha

    def train(self, X, y, num_iters=1000):
        N, d = X.shape
        
        # YOUR CODE HERE
        self.w = np.zeros(d)
        # Precompute column norms squared: ||X_j||^2
        col_norms_sq = np.sum(X ** 2, axis=0)  # (d,)
        
        for _ in range(num_iters):
            for j in range(d):
                # Partial residual: y - X @ w + X[:,j] * w[j]
                r_j = y - X @ self.w + X[:, j] * self.w[j]
                # Correlation
                rho_j = X[:, j] @ r_j
                # Coordinate update with soft threshold
                if col_norms_sq[j] > 1e-10:
                    self.w[j] = soft_threshold(rho_j / col_norms_sq[j], self.alpha / col_norms_sq[j])

    def predict(self, X): 
        return X @ self.w

In [ ]:
model = LassoCoordinateDescent(alpha=0.1)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

In [ ]:
model = LassoCoordinateDescent(alpha=2)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

---
## Récapitulatif comparatif des 4 méthodes

| Méthode | Différentiabilité | Sparsité exacte | Convergence | Learning rate | Intercept |
|---|---|---|---|---|---|
| **Subgradient Descent** | Non (sous-gradient) | Non (approx.) | $O(1/\sqrt{t})$ | Requis | Oui |
| **Proximal GD (ISTA)** | Lisse + proximal | Oui | $O(1/t)$ | Requis | Oui |
| **Projected GD** | Oui (après décomposition) | Oui | $O(1/t)$ | Requis | Oui |
| **Coordinate Descent** | Oui (par coordonnée) | Oui (exacte) | Garantie globale | Non requis | Non |

### Recommandation pratique

- Pour des grands problèmes : **Coordinate Descent** (utilisé par scikit-learn)
- Pour du stochastique (big data) : **Proximal SGD (ISTA)**
- Le Subgradient est le plus simple mais le moins précis